# 02 Feature Engineering v1

Purpose: build one modeling table with one row per user. Version 1 uses labels, member features, and transaction features only. User logs are excluded for scope control.

## Setup And Load Data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\muham\OneDrive\Desktop\ML Journey\Project\kkbox-churn")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

TRAIN_PATH = RAW_DIR / "train_v2.csv"
MEMBERS_PATH = RAW_DIR / "members_v3.csv"
TRANSACTIONS_PATH = RAW_DIR / "transactions.csv"
CUTOFF_DATE = pd.Timestamp("2017-02-28")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(TRAIN_PATH)
members = pd.read_csv(MEMBERS_PATH)

print("Train shape:", train.shape)
print("Members shape:", members.shape)

## Member Features

Clean age, handle missing gender, create account tenure, and mark future registration dates. Future registration-derived tenure is set to missing to avoid leakage.

In [ ]:
member_features = members.copy()
member_features["has_member_data"] = 1
member_features["age_clean"] = member_features["bd"].where(member_features["bd"].between(13, 80), np.nan)
member_features["age_missing_or_invalid"] = member_features["age_clean"].isna().astype(int)
member_features["gender"] = member_features["gender"].fillna("unknown")
member_features["registration_init_time_parsed"] = pd.to_datetime(
    member_features["registration_init_time"].astype(str), format="%Y%m%d", errors="coerce"
)
member_features["registration_after_cutoff_flag"] = (member_features["registration_init_time_parsed"] > CUTOFF_DATE).astype(int)
member_features["registration_tenure_days"] = (CUTOFF_DATE - member_features["registration_init_time_parsed"]).dt.days
member_features.loc[member_features["registration_after_cutoff_flag"] == 1, "registration_tenure_days"] = np.nan

member_features = member_features[
    ["msno", "has_member_data", "age_clean", "age_missing_or_invalid", "gender", "city",
     "registered_via", "registration_tenure_days", "registration_after_cutoff_flag"]
]

print("Member features shape:", member_features.shape)
print(member_features.isna().sum())

## Join Member Features

Use a left join so all labeled users remain. Missing member rows are treated separately from missing values inside existing member rows.

In [ ]:
train_with_member_features = train.merge(member_features, on="msno", how="left")
train_with_member_features["has_member_data"] = train_with_member_features["has_member_data"].fillna(0).astype(int)
train_with_member_features["gender"] = train_with_member_features["gender"].fillna("no_member_data")
train_with_member_features["city"] = train_with_member_features["city"].fillna(-999).astype(int)
train_with_member_features["registered_via"] = train_with_member_features["registered_via"].fillna(-999).astype(int)
train_with_member_features["age_missing_or_invalid"] = train_with_member_features["age_missing_or_invalid"].fillna(1).astype(int)
train_with_member_features["registration_after_cutoff_flag"] = train_with_member_features["registration_after_cutoff_flag"].fillna(0).astype(int)

print(train_with_member_features.isna().sum())
print(train_with_member_features.groupby("has_member_data")["is_churn"].mean())

## Transaction Features

Aggregate many transaction rows into user-level features. Also keep the latest transaction because recent behavior is often highly predictive.

In [ ]:
transaction_cols = [
    "msno", "payment_method_id", "payment_plan_days", "plan_list_price",
    "actual_amount_paid", "is_auto_renew", "transaction_date",
    "membership_expire_date", "is_cancel"
]
transactions = pd.read_csv(TRANSACTIONS_PATH, usecols=transaction_cols)

transactions["transaction_date_parsed"] = pd.to_datetime(transactions["transaction_date"].astype(str), format="%Y%m%d", errors="coerce")
transactions["membership_expire_date_parsed"] = pd.to_datetime(transactions["membership_expire_date"].astype(str), format="%Y%m%d", errors="coerce")
transactions["discount_amount"] = transactions["plan_list_price"] - transactions["actual_amount_paid"]
transactions["is_zero_payment_or_plan"] = (
    (transactions["payment_plan_days"] == 0)
    | (transactions["plan_list_price"] == 0)
    | (transactions["actual_amount_paid"] == 0)
).astype(int)

summary = transactions.groupby("msno").agg(
    transaction_count=("msno", "size"),
    cancel_count=("is_cancel", "sum"),
    auto_renew_count=("is_auto_renew", "sum"),
    zero_payment_or_plan_count=("is_zero_payment_or_plan", "sum"),
    unique_payment_method_count=("payment_method_id", "nunique"),
    total_actual_amount_paid=("actual_amount_paid", "sum"),
    mean_actual_amount_paid=("actual_amount_paid", "mean"),
    max_actual_amount_paid=("actual_amount_paid", "max"),
    mean_plan_list_price=("plan_list_price", "mean"),
    mean_payment_plan_days=("payment_plan_days", "mean"),
    discount_transaction_count=("discount_amount", lambda x: (x > 0).sum()),
).reset_index()
summary["manual_renew_count"] = summary["transaction_count"] - summary["auto_renew_count"]

latest = transactions.sort_values(["msno", "transaction_date_parsed", "membership_expire_date_parsed"]).groupby("msno").tail(1).copy()
latest = latest.rename(columns={
    "payment_method_id": "latest_payment_method_id",
    "payment_plan_days": "latest_payment_plan_days",
    "plan_list_price": "latest_plan_list_price",
    "actual_amount_paid": "latest_actual_amount_paid",
    "is_auto_renew": "latest_is_auto_renew",
    "is_cancel": "latest_is_cancel",
    "discount_amount": "latest_discount_amount",
})
latest["latest_transaction_days_before_cutoff"] = CUTOFF_DATE - latest["transaction_date_parsed"]
latest["latest_transaction_days_before_cutoff"] = latest["latest_transaction_days_before_cutoff"].dt.days
latest["latest_membership_expire_days_after_cutoff"] = (latest["membership_expire_date_parsed"] - CUTOFF_DATE).dt.days
latest = latest[[
    "msno", "latest_payment_method_id", "latest_payment_plan_days", "latest_plan_list_price",
    "latest_actual_amount_paid", "latest_is_auto_renew", "latest_is_cancel", "latest_discount_amount",
    "latest_transaction_days_before_cutoff", "latest_membership_expire_days_after_cutoff"
]]

transaction_features = summary.merge(latest, on="msno", how="left")
print("Transaction features shape:", transaction_features.shape)

## Final Modeling Table

Join transaction features to the label-member table. Keep all labeled users. Missing transaction features mean no transaction history and should be encoded deliberately.

In [ ]:
modeling_data_v1 = train_with_member_features.merge(transaction_features, on="msno", how="left")

count_cols = [
    "transaction_count", "cancel_count", "auto_renew_count", "zero_payment_or_plan_count",
    "unique_payment_method_count", "total_actual_amount_paid", "mean_actual_amount_paid",
    "max_actual_amount_paid", "mean_plan_list_price", "mean_payment_plan_days",
    "discount_transaction_count", "manual_renew_count"
]
latest_num_cols = [
    "latest_payment_plan_days", "latest_plan_list_price", "latest_actual_amount_paid",
    "latest_discount_amount", "latest_transaction_days_before_cutoff",
    "latest_membership_expire_days_after_cutoff"
]
latest_cat_cols = ["latest_payment_method_id", "latest_is_auto_renew", "latest_is_cancel"]

modeling_data_v1["has_transaction_data"] = modeling_data_v1["transaction_count"].notna().astype(int)
modeling_data_v1[count_cols] = modeling_data_v1[count_cols].fillna(0)
modeling_data_v1[latest_num_cols] = modeling_data_v1[latest_num_cols].fillna(-999)
modeling_data_v1[latest_cat_cols] = modeling_data_v1[latest_cat_cols].fillna(-999).astype(int)

print("Final shape:", modeling_data_v1.shape)
print("Duplicate users:", modeling_data_v1["msno"].duplicated().sum())
print("Target rate:")
print(modeling_data_v1["is_churn"].value_counts(normalize=True).round(4))
print(modeling_data_v1.isna().sum())

output_path = PROCESSED_DIR / "kkbox_modeling_data_v1.csv"
modeling_data_v1.to_csv(output_path, index=False)
print("Saved:", output_path)

### Interpretation

The final modeling table should preserve the original number of users and churn rate. If joins change the target rate or create duplicate users, the feature engineering is broken.